# DB API EXTRACTION

In [ ]:
import requests

BASE_URL = "https://api.hackupm2025.workers.dev/api/v1/patients/train"
LIMIT = 100

def fetch_all_patients():
    page = 1
    all_patients = []

    while True:
        params = {"page": page, "limit": LIMIT}
        response = requests.get(BASE_URL, params=params)

        # Raise for bad responses (4xx, 5xx)
        response.raise_for_status()

        data = response.json()

        # Extract the patients from the 'data' field
        patients = data.get("data", [])
        all_patients.extend(patients)

        # Extract pagination info
        pagination = data.get("pagination", {})
        has_next = pagination.get("hasNextPage", False)

        print(f"Fetched page {page} → {len(patients)} records")

        if not has_next:
            break  # Stop when no more pages
        page += 1

    print(f"\n✅ Done! Total patients fetched: {len(all_patients)}")
    return all_patients


if __name__ == "__main__":
    all_data = fetch_all_patients()

    # Example: write results to a JSON file
    import json
    filepath_out = "../0. Datasets/patients_data.json"
    with open(filepath_out, "w", encoding="utf-8") as f:
        json.dump(all_data, f, ensure_ascii=False, indent=2)

    print("💾 Saved all patients to patients_data.json")


Fetched page 1 → 100 records
Fetched page 2 → 100 records
Fetched page 3 → 100 records
Fetched page 4 → 100 records
Fetched page 5 → 100 records
Fetched page 6 → 100 records
Fetched page 7 → 100 records
Fetched page 8 → 100 records
Fetched page 9 → 100 records
Fetched page 10 → 100 records
Fetched page 11 → 100 records
Fetched page 12 → 100 records
Fetched page 13 → 100 records
Fetched page 14 → 100 records
Fetched page 15 → 100 records
Fetched page 16 → 100 records
Fetched page 17 → 100 records
Fetched page 18 → 100 records
Fetched page 19 → 100 records
Fetched page 20 → 100 records
Fetched page 21 → 100 records
Fetched page 22 → 100 records
Fetched page 23 → 100 records
Fetched page 24 → 100 records
Fetched page 25 → 100 records
Fetched page 26 → 100 records
Fetched page 27 → 100 records
Fetched page 28 → 100 records
Fetched page 29 → 100 records
Fetched page 30 → 100 records

✅ Done! Total patients fetched: 3000
💾 Saved all patients to patients_data.json


# DB NLP EXTRACTION

NOTE: TRANSLATE MEDICAL_NOTES TO ENGLISH????

In [7]:
#First step: Load the JSON data
import json
import pandas as pd

filepath_in = "../0. Datasets/patients_data.json"
with open(filepath_in, "r") as f:
    data = json.load(f)

In [14]:
#Second step: Define extract functions
import re
from word2number import w2n

def extract_age(note):
    # Numeric ages first
    match = re.search(r'(\d+)[ -]?year[s]?[- ]?old', note, re.I)
    if match:
        return int(match.group(1))
    
    # Written ages
    match_word = re.search(r'([a-z- ]+)[ -]?year[s]?[- ]?old', note, re.I)
    if match_word:
        try:
            return w2n.word_to_num(match_word.group(1))
        except ValueError:
            return None
    return None

def extract_gender(note):
    text = note.lower()

    # --- FEMALE ---
    if re.search(r'\b(female|girl|woman)\b', text):
        return "Female"
    # look for standalone 'she' (with whitespace or punctuation around)
    if re.search(r'(?<![a-z])she(?![a-z])', text):
        return "Female"

    # --- MALE ---
    # explicit words first, but make sure 'man' isn't part of 'woman'
    if re.search(r'\bmale\b', text) or re.search(r'\b(boy|man)\b(?!\s*and\s*woman)', text):
        return "Male"
    # standalone 'he' — only if 'she' isn't also found
    if re.search(r'(?<![a-z])he(?![a-z])', text) and not re.search(r'(?<![a-z])she(?![a-z])', text):
        return "Male"

    return None

# Simplified: detects mention and negation within the same sentence
def extract_hypertension(note):
    if not note:
        return None

    text = note.lower()

    # List of target keywords and negations
    keywords = ["hyperten"]  # matches 'hypertension', 'hypertensive', etc.
    negations = ["no", "not", "without", "denies", "none", "absence of", "free of"]

    keyword_found = False

    for key in keywords:
        # Find all occurrences of the keyword
        for match in re.finditer(rf'\b{re.escape(key)}\w*\b', text):
            keyword_found = True
            # Look at a window of ~50 chars before the keyword for negations
            start_idx = max(0, match.start() - 50)
            context = text[start_idx:match.start()]

            if any(re.search(rf'\b{re.escape(neg)}\b', context) for neg in negations):
                return False

    if keyword_found:
        return True
    else:
        return None

def extract_heart_disease(note):
    """
    Detects whether heart disease is present in a clinical note.

    Returns:
        True  -> condition present
        False -> condition negated
        None  -> condition not mentioned
    """
    if not note:
        return None

    text = note.lower()

    # List of target keywords and negations
    keywords = ["heart", "cardiac", "myocardial", "coronary"]  # add variations if needed
    negations = ["no", "not", "without", "denies", "none", "absence of", "free of"]

    keyword_found = False

    for key in keywords:
        # Find all occurrences of the keyword
        for match in re.finditer(rf'\b{re.escape(key)}\w*\b', text):
            keyword_found = True
            # Look at a window of ~50 characters before the keyword for negations
            start_idx = max(0, match.start() - 50)
            context = text[start_idx:match.start()]

            if any(re.search(rf'\b{re.escape(neg)}\b', context) for neg in negations):
                return False

    if keyword_found:
        return True
    else:
        return None

def extract_smoking(note):
    """
    Detects smoking status from a clinical note based on keywords and negations.

    Returns:
        "Non-smoker" -> explicitly negated or never smoked
        "Current"    -> currently smoking
        "Past"       -> former smoker
        None         -> not mentioned or unclear
    """
    if not note:
        return None

    text = note.lower()
    
    # Directly detect "non-smok" anywhere in the text
    if "non-smok" in text:
        return "Non-smoker"

    keywords = ["smok"]  # matches 'smoke', 'smoker', 'smoking', etc.
    negations = ["no", "not", "never", "denies", "without", "none", "free of", "absence of"]
    past_terms = ["past", "former", "history of"]

    for key in keywords:
        for match in re.finditer(rf'\b{re.escape(key)}\w*\b', text):
            # Only look back 30 characters before the keyword for negations
            start_idx = max(0, match.start() - 50)
            context = text[start_idx:match.start()]

            # Check for negations in context
            if any(re.search(rf'\b{re.escape(neg)}\b', context) for neg in negations):
                return "Non-smoker"

            # Check for past smoker terms in the **whole keyword word** (can adjust later if needed)
            start_idx = max(0, match.start() - 15)
            end_idx = match.end() + 15
            keyword_context = text[start_idx:end_idx] #look ahead and behind (context window of 15 for both)
            if any(re.search(rf'\b{re.escape(term)}\b', keyword_context) for term in past_terms):
                return "Past"

            # Otherwise assume current smoker
            return "Current"

    return None

def extract_bmi(note):
    """
    Extracts BMI from a clinical note in the format 'BMI of X' where X is a float.
    
    Returns:
        float: The BMI value if found.
        None: If no BMI pattern is detected.
    """
    #match = re.search(r'BMI\s+of\s+([\d]+(?:\.\d+)?)', note, re.I)
    match = re.search(r'(?:BMI|Body Mass Index)[^0-9]*([\d]+(?:\.\d+)?)', note, re.I)
    return float(match.group(1)) if match else None


def extract_hba1c_pure_float(note):
    """
    Extracts HbA1c value from a clinical note by detecting 'HbA1c'
    and capturing the next float within 15 characters forward.

    Returns:
        float: The HbA1c value if found.
        None:  If no valid pattern is detected.
    """
    if not note:
        return None

    # Search for 'HbA1c' (case-insensitive)
    match = re.search(r'HbA1c', note, re.I)
    if not match:
        return None

    # Look 25 characters forward from the end of the match
    start_idx = match.end()
    end_idx = start_idx + 25
    context = note[start_idx:end_idx]

    # Find the first float number in that context
    value_match = re.search(r'([\d]+(?:\.\d+)?)', context)
    return float(value_match.group(1)) if value_match else None

def extract_glucose_pure_float(note):
    """
    Extracts glucose value from a clinical note by detecting 'glucose'
    and capturing the next float within 15 characters forward.

    Returns:
        float: The glucose value if found.
        None:  If no valid pattern is detected.
    """
    if not note:
        return None

    # Search for 'glucose' (case-insensitive)
    match = re.search(r'glucose', note, re.I)
    if not match:
        return None

    # Look 25 characters forward from the end of the match
    start_idx = match.end()
    end_idx = start_idx + 25
    context = note[start_idx:end_idx]

    # Find the first float number in that context
    value_match = re.search(r'([\d]+(?:\.\d+)?)', context)
    return float(value_match.group(1)) if value_match else None

# ==================================
# ----------------------------------
# ==================================

def extract_hba1c_qualitative(note):
    """
    Extracts qualitative HbA1c descriptions from a clinical note and 
    converts them into representative float values based on medical meaning.

    Returns:
        float: Representative HbA1c value if a qualitative term is found.
        None:  If no valid qualitative description is detected.
    """
    if not note:
        return None

    # Search for 'HbA1c' (case-insensitive)
    match = re.search(r'HbA1c', note, re.I)
    if not match:
        return None

    # Look 30 characters forward from the end of the match and 15 backwards from start of match
    start_idx = max(0, match.start() - 15)
    end_idx = match.end() + 30
    context = note[start_idx:end_idx].lower()


    # Detect qualitative descriptors and map to representative numeric values
    if re.search(r'very\s+(high|elevated)', context):
        return 8.8
    elif re.search(r'\b(high|elevated)\b', context):
        return 6.8
    elif re.search(r'\bnormal\b', context):
        return 5.3
    elif re.search(r'very\s+low', context):
        return 4.3
    elif re.search(r'\blow\b', context):
        return 4.8

    return None


def extract_glucose_qualitative(note):
    """
    Extracts qualitative Random Glucose descriptions from a clinical note and 
    converts them into representative float values based on medical meaning.

    Returns:
        float: Representative glucose value if a qualitative term is found.
        None:  If no valid qualitative description is detected.
    """
    if not note:
        return None

    # Search for 'glucose' (case-insensitive)
    match = re.search(r'glucose', note, re.I)
    if not match:
        return None

    # Look 30 characters forward from the end of the match and 15 backwards from start of match
    start_idx = max(0, match.start() - 15)
    end_idx = match.end() + 30
    context = note[start_idx:end_idx].lower()

    # Detect qualitative descriptors and map to representative numeric values
    if re.search(r'very\s+(high|elevated)', context):
        return 240.0
    elif re.search(r'\b(high|elevated)\b', context):
        return 160.0
    elif re.search(r'\bnormal\b', context):
        return 100.0
    elif re.search(r'very\s+low', context):
        return 55.0
    elif re.search(r'\blow\b', context):
        return 70.0

    return None

In [15]:
# --- Apply extraction and create structured DataFrame ---
from tqdm import tqdm
import re

records = []

for p in tqdm(data):
    note = p.get("medical_note", "")
    sentences = re.split(r'(?<=[.!?])\s+', note.strip())  # Split into sentences safely

    # Initialize patient record (keep consistent order)
    extracted = {
        "patient_id": p.get("patient_id"),
        "Heart Disease": None,
        "Age": None,
        "Gender": None,
        "Hypertension": None,
        "Smoking History": None,
        "BMI": None,
        "HbA1c": None,
        "Random Glucose": None,
        "has_diabetes": p.get("has_diabetes"),
    }

    # Iterate through sentences — stop when all found
    for sentence in sentences:
        if not sentence.strip():
            continue

        if extracted["Age"] is None:
            extracted["Age"] = extract_age(sentence)
        if extracted["Gender"] is None:
            extracted["Gender"] = extract_gender(sentence)
        if extracted["Hypertension"] is None:
            extracted["Hypertension"] = extract_hypertension(sentence)
        if extracted["Heart Disease"] is None:
            extracted["Heart Disease"] = extract_heart_disease(sentence)
        if extracted["Smoking History"] is None:
            extracted["Smoking History"] = extract_smoking(sentence)
        if extracted["BMI"] is None:
            extracted["BMI"] = extract_bmi(sentence)
        
        # --- Try numeric first
        if extracted["HbA1c"] is None:
            extracted["HbA1c"] = extract_hba1c_pure_float(sentence)
        if extracted["Random Glucose"] is None:
            extracted["Random Glucose"] = extract_glucose_pure_float(sentence)
            
        # --- Then qualitative fallback
        if extracted["HbA1c"] is None:
            extracted["HbA1c"] = extract_hba1c_qualitative(sentence)
        if extracted["Random Glucose"] is None:
            extracted["Random Glucose"] = extract_glucose_qualitative(sentence)

        # Stop early if everything found
        if all(v is not None for k, v in extracted.items() if k not in ["patient_id", "has_diabetes"]):
            break

    records.append(extracted)  # ✅ only one record per patient

# --- Convert to DataFrame ---
df = pd.DataFrame(records, columns=[
    "patient_id",
    "has_diabetes",
    "Age",
    "Gender",
    "Hypertension",
    "Heart Disease",
    "Smoking History",
    "BMI",
    "HbA1c",
    "Random Glucose",
])

print(f"✅ Processed {len(df)} patients.")
display(df.head(20))

100%|██████████| 3000/3000 [00:00<00:00, 15106.12it/s]


✅ Processed 3000 patients.


,patient_id,has_diabetes,Age,Gender,Hypertension,Heart Disease,Smoking History,BMI,HbA1c,Random Glucose
0,82555,0,16.0,Female,False,False,Non-smoker,21.49,6.2,100.0
1,92299,0,15.0,Female,False,False,Non-smoker,33.62,5.3,158.0
2,18725,0,54.0,Male,False,False,Current,21.46,5.3,145.0
3,52208,1,54.0,Female,False,False,Non-smoker,39.12,6.8,1.0
4,2640,0,29.0,Female,False,False,Past,38.25,3.5,126.0
5,10066,0,34.0,Female,False,False,Non-smoker,21.29,6.2,140.0
6,46513,0,66.0,Female,False,False,Past,27.32,4.8,100.0
7,78041,0,15.0,Female,False,False,Non-smoker,24.62,5.0,100.0
8,73052,1,71.0,Male,False,True,Past,31.01,8.8,145.0
9,34838,0,80.0,Female,False,True,Non-smoker,27.30,5.3,90.0


In [16]:
#Fourth step: Save to CSV file
filepath_out = "../0. Datasets/patients_structured.csv"

df.to_csv(filepath_out, index=False)